# <center> Data Flow City Staging Table </center>
```mermaid
flowchart TB
Address_City --> city_city
states_Name --> city_state_province
states_StateProvinceCode --> city_subregion
CountryRegion_Name --> city_country
SalesTerritory_Name --> city_sales_territory
SalesTerritory_CountryRegionCode --> city_region
SalesTerritory_Group --> city_continent

city_city_staging_key ins1@-- UPSERT --> Dim_city_city_staging_key
city_city ins2@-- UPSERT --> Dim_city_city
city_state_province ins3@-- UPSERT --> Dim_city_state_province
city_country ins4@-- UPSERT --> Dim_city_country
city_continent ins5@-- UPSERT --> Dim_city_continent
city_sales_territory ins6@-- UPSERT --> Dim_city_sales_territory
city_region ins7@-- UPSERT --> Dim_city_region
city_subregion ins8@-- UPSERT --> Dim_city_subregion
ins1@{animation: fast}
ins2@{animation: fast}
ins3@{animation: fast}
ins4@{animation: fast}
ins5@{animation: fast}
ins6@{animation: fast}
ins7@{animation: fast}
ins8@{animation: fast}

Address_StateProvinceID j1@o-.JOIN.-o states_StateProvinceID
j1@{animation: slow}
SalesTerritory_TerritoryID j3@o-.JOIN.-o states_TerritoryID
j3@{animation: slow}
CountryRegion_CountryRegionCode j2@o-.JOIN.-o states_CountryRegionCode
j2@{animation: slow}

    subgraph city staging
        direction LR
        city_city_staging_key[City Staging Key] 
        city_city[City] 
        city_state_province[State Province] 
        city_country[Country] 
        city_continent[Continent] 
        city_sales_territory[Sales Territory] 
        city_region[Region] 
        city_subregion[Subregion] 
         
    end
    subgraph Dimension.City
        direction LR
        Dim_city_city_staging_key[City Staging Key] 
        Dim_city_city[City] 
        Dim_city_state_province[State Province] 
        Dim_city_country[Country] 
        Dim_city_continent[Continent] 
        Dim_city_sales_territory[Sales Territory] 
        Dim_city_region[Region] 
        Dim_city_subregion[Subregion] 
         
    end
    subgraph Source
        subgraph Person.Address
            direction LR
            Address_StateProvinceID[StateProvinceID]
            Address_City[City]
        end
        subgraph Person.StateProvince
            direction LR
            states_Name[Name]
            states_StateProvinceCode[StateProvinceCode] 
            states_StateProvinceID[StateProvinceID] 
            states_TerritoryID[TerritoryID] 
            states_CountryRegionCode[CountryRegionCode] 
        end
        subgraph Person.CountryRegion
            direction LR
            CountryRegion_CountryRegionCode[CountryRegionCode]
            CountryRegion_Name[Country]
        end
        subgraph Sales.SalesTerritory
            direction LR
            SalesTerritory_TerritoryID
            SalesTerritory_Name[Name]
            SalesTerritory_CountryRegionCode[CountryRegionCode]
            SalesTerritory_Group[Group]
        end
    end
```

## <center>Data Collection</center>

#### Connection

In [1]:
from pyspark.sql import functions as sf
from pyspark.sql import types as sdt
from pyspark.sql import SparkSession
from datetime import datetime


In [2]:

LOCAL_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_warehouse"
STG_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_staging_warehouse"
RPT_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_data_warehouse"
CATALOG_NAME = "local"
STG_CATALOG_NAME = "staging"
WH_CATALOG_NAME = "reporting"

spark = SparkSession.builder \
    .appName("Iceberg Setup") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", f"file:///{LOCAL_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.warehouse", f"file:///{STG_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.warehouse", f"file:///{RPT_WAREHOUSE_PATH}") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .getOrCreate()


spark.catalog.setCurrentCatalog("local")

spark

#### Table collection

In [3]:
Address = spark.table("local.Person.Address").select("City", "StateProvinceID",).distinct().alias("cities")
StateProvince = spark.table("local.Person.StateProvince").select("Name", "StateProvinceCode", "StateProvinceID", "TerritoryID", "CountryRegionCode",).alias("states")
SalesTerritory = spark.table("local.Sales.SalesTerritory").select("Group", "Name", "CountryRegionCode", "TerritoryID",).alias("territories")
CountryRegion = spark.table("local.Person.CountryRegion").select("Name", "CountryRegionCode", ).alias("countries")

Address.show(5, truncate=False)
StateProvince.show(5, truncate=False)
SalesTerritory.show(5, truncate=False)
CountryRegion.show(5, truncate=False)



+--------------+---------------+
|City          |StateProvinceID|
+--------------+---------------+
|Everett       |79             |
|Spring Valley |9              |
|Kennewick     |79             |
|Ontario       |9              |
|Trabuco Canyon|9              |
+--------------+---------------+
only showing top 5 rows
+--------------+-----------------+---------------+-----------+-----------------+
|Name          |StateProvinceCode|StateProvinceID|TerritoryID|CountryRegionCode|
+--------------+-----------------+---------------+-----------+-----------------+
|Alberta       |AB               |1              |6          |CA               |
|Alaska        |AK               |2              |1          |US               |
|Alabama       |AL               |3              |5          |US               |
|Arkansas      |AR               |4              |3          |US               |
|American Samoa|AS               |5              |1          |AS               |
+--------------+---------------

#### <center>Create table on **iceberg**</center>

In [4]:
sql_Integration_City = "DROP TABLE IF EXISTS staging.Integration.City "

spark.sql(sql_Integration_City)

sql_Integration_City = "CREATE TABLE IF NOT EXISTS staging.Integration.City ( "\
    "city_staging_key INT COMMENT 'Row ID within the staging table', "\
    "city STRING COMMENT 'Formal name of the city', "\
    "state_province STRING COMMENT 'State or province for this city', "\
    "country STRING COMMENT 'Country name', "\
    "continent STRING COMMENT 'Continent that this city is on', "\
    "sales_territory STRING COMMENT 'Sales territory for this StateProvince', "\
    "region STRING COMMENT 'Name of the region', "\
    "sub_region STRING COMMENT 'Name of the subregion', "\
    "valid_from TIMESTAMP COMMENT 'Valid from this date and time', "\
    "valid_to TIMESTAMP COMMENT 'Valid until this date and time' ,"\
    "record_hash STRING COMMENT 'Hash of tracked columns for Type 2 change detection'"\
    ") "\
"USING ICEBERG "\
"PARTITIONED BY (country, state_province) "\
"TBLPROPERTIES ('comment' = 'Staging table for city-level integration data')"

spark.sql(sql_Integration_City)

DataFrame[]

#### Table Joining

In [5]:

joined_output = Address\
    .join(StateProvince, sf.col("cities.StateProvinceID") == sf.col("states.StateProvinceID"),how="inner")\
    .join(SalesTerritory, sf.col("states.TerritoryID") == sf.col("territories.TerritoryID"),how="inner")\
    .join(CountryRegion, sf.col("states.CountryRegionCode") == sf.col("countries.CountryRegionCode"),how="inner")\
    .orderBy("territories.Group", "countries.Name", "states.Name", "cities.City", )\
    .select(
        sf.col("cities.City").alias("City"),
        sf.col("states.Name").alias("State_Province"),
        sf.col("countries.Name").alias("Country"),
        sf.col("territories.Group").alias("Continent"),
        sf.col("territories.Name").alias("Sales_Territory"),
        sf.col("territories.CountryRegionCode").alias("Region"),
        sf.col("states.StateProvinceCode").alias("Sub_Region"),
    )
joined_output = joined_output \
    .withColumn("city_staging_key", sf.monotonically_increasing_id().cast("int") + 1) \
    .withColumn("valid_from", sf.lit(datetime.now())) \
    .withColumn("valid_to", sf.lit(None).cast("timestamp"))\
    .withColumn("record_hash",
                sf.sha2(sf.concat_ws("||",
                sf.col("Continent"),
                sf.col("Sales_Territory"),
                sf.col("Region"),
                sf.col("Sub_Region")
                ), 256)
                )


joined_output = joined_output.select(
        "city_staging_key",
        "City",
        "State_Province",
        "Country",
        "Continent",
        "Sales_Territory",
        "Region",
        "Sub_Region",
        "valid_from",
        "valid_to",
        "record_hash",
    )
joined_output = joined_output.select(
    sf.col("city_staging_key"),
    sf.col("City").alias("city"),
    sf.col("State_Province").alias("state_province"),
    sf.col("Country").alias("country"),
    sf.col("Continent").alias("continent"),
    sf.col("Sales_Territory").alias("sales_territory"),
    sf.col("Region").alias("region"),
    sf.col("Sub_Region").alias("sub_region"),
    sf.col("valid_from"),
    sf.col("valid_to"),
    sf.col("record_hash")
)


joined_output.show(5, truncate=False)

+----------------+--------------------+-----------------+-------+---------+---------------+------+----------+--------------------------+--------+----------------------------------------------------------------+
|city_staging_key|city                |state_province   |country|continent|sales_territory|region|sub_region|valid_from                |valid_to|record_hash                                                     |
+----------------+--------------------+-----------------+-------+---------+---------------+------+----------+--------------------------+--------+----------------------------------------------------------------+
|1               |Saint Ouen          |Charente-Maritime|France |Europe   |France         |FR    |17        |2025-11-14 14:21:07.692407|NULL    |d8ca5784376e27edaee76a491ab6e92bff2ed5db3380d0098b66a3a68432b81e|
|2               |Les Ulis            |Essonne          |France |Europe   |France         |FR    |91        |2025-11-14 14:21:07.692407|NULL    |55db1ce35d1

#### Insert/load into Table

In [6]:

# Step 5: Write to Iceberg table
joined_output.writeTo("staging.Integration.City")\
    .using("iceberg")\
    .overwritePartitions()


### See Table Properties

In [7]:
spark.sql("SHOW TBLPROPERTIES staging.Integration.City").show(truncate=False)
spark.sql("DESCRIBE TABLE EXTENDED staging.Integration.City").show(truncate=False)
spark.read.table("staging.Integration.City.partitions").show(truncate=False)

+-------------------------------+-------------------+
|key                            |value              |
+-------------------------------+-------------------+
|current-snapshot-id            |6869414015612046087|
|format                         |iceberg/parquet    |
|format-version                 |2                  |
|write.parquet.compression-codec|zstd               |
+-------------------------------+-------------------+

+-----------------------+--------------------------------------------+---------------------------------------------------+
|col_name               |data_type                                   |comment                                            |
+-----------------------+--------------------------------------------+---------------------------------------------------+
|city_staging_key       |int                                         |Row ID within the staging table                    |
|city                   |string                                      |Forma

In [8]:
Integration_City = spark.table("staging.Integration.City")\
    .select(
        "city_staging_key", 
        "city", 
        "state_province", 
        "country", 
        "continent", 
        "sales_territory", 
        "region", 
        "sub_region", 
        "valid_from", 
        "valid_to",
        "record_hash",
    )
Integration_City.printSchema()

Integration_City\
    .orderBy("city_staging_key")\
    .show(10,truncate=False)



root
 |-- city_staging_key: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- state_province: string (nullable = true)
 |-- country: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- sales_territory: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sub_region: string (nullable = true)
 |-- valid_from: timestamp (nullable = true)
 |-- valid_to: timestamp (nullable = true)
 |-- record_hash: string (nullable = true)

+----------------+--------------------+-----------------+-------+---------+---------------+------+----------+--------------------------+--------+----------------------------------------------------------------+
|city_staging_key|city                |state_province   |country|continent|sales_territory|region|sub_region|valid_from                |valid_to|record_hash                                                     |
+----------------+--------------------+-----------------+-------+---------+---------------+------+-----

#### 	Table Dimension.City

In [9]:
sql_dimension_City = "CREATE TABLE IF NOT EXISTS reporting.dimension.City ( "\
    "city_key INT COMMENT 'DW key for the city dimension', "\
    "city STRING COMMENT 'Formal name of the city', "\
    "state_province STRING COMMENT 'State or province for this city', "\
    "country STRING COMMENT 'Country name', "\
    "continent STRING COMMENT 'Continent that this city is on', "\
    "sales_territory STRING COMMENT 'Sales territory for this StateProvince', "\
    "region STRING COMMENT 'Name of the region', "\
    "sub_region STRING COMMENT 'Name of the subregion', "\
    "valid_from TIMESTAMP COMMENT 'Valid from this date and time', "\
    "valid_to TIMESTAMP COMMENT 'Valid until this date and time', "\
    "record_hash STRING COMMENT 'Hash of tracked columns for Type 2 change detection' "\
    ") "\
"USING ICEBERG "\
"PARTITIONED BY (region, sub_region) "\
"TBLPROPERTIES ('comment' = 'dimension table for city-level reporting data')"

# sql_dimension_City = "DROP TABLE reporting.dimension.City"

spark.sql(sql_dimension_City)

DataFrame[]

In [10]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

dest_table = spark.table("reporting.dimension.City")\
    .select(
        "record_hash",
        "city_key",
        ).alias("dest")

raw_key = dest_table.agg(sf.max("city_key")).collect()[0][0]
last_staging_key = raw_key if raw_key is not None else 0

scrc_table = spark.table("staging.Integration.City")\
    .select(
        "city",
        "state_province",
        "country",
        "continent",
        "sales_territory",
        "region",
        "sub_region",
        "valid_from",
        "valid_to",
        "record_hash",
        ).alias("scrc")


insert_rec = scrc_table\
    .join(dest_table, sf.col("scrc.record_hash") == sf.col("dest.record_hash"), how="left")\
    .filter(dest_table["record_hash"].isNull())\
    .select("scrc.*")\

    # .filter("dest.record_hash is Null")\
# insert_rec = insert_rec \
#     .withColumn("city_key", (sf.monotonically_increasing_id() + 1 + last_staging_key).cast("int")) \

window_spec = Window.orderBy("city", "state_province", "country")

insert_rec = insert_rec.withColumn(
  "city_key",
  row_number().over(window_spec) + last_staging_key
)


insert_rec = insert_rec.select(
    "city_key",
    "city",
    "state_province",
    "country",
    "continent",
    "sales_territory",
    "region",
    "sub_region",
    "valid_from",
    "valid_to",
    "record_hash",
)

# insert_rec.show(10, truncate=False)

# insert_rec.printSchema()
    
insert_rec.writeTo("reporting.dimension.City")\
    .using("iceberg")\
    .overwritePartitions()





In [11]:
dimension_City = spark.table("reporting.dimension.City")\
    .select(
        "city_key", 
        "city", 
        "state_province", 
        "country", 
        "continent", 
        "sales_territory", 
        "region", 
        "sub_region", 
        "valid_from", 
        "valid_to",
        "record_hash",
    )
print(dimension_City.count())

dimension_City\
    .orderBy("city_key")\
    .show(10,truncate=False)



613
+--------+-----------------+---------------+--------------+-------------+---------------+------+----------+--------------------------+--------+----------------------------------------------------------------+
|city_key|city             |state_province |country       |continent    |sales_territory|region|sub_region|valid_from                |valid_to|record_hash                                                     |
+--------+-----------------+---------------+--------------+-------------+---------------+------+----------+--------------------------+--------+----------------------------------------------------------------+
|614     |Abingdon         |England        |United Kingdom|Europe       |United Kingdom |GB    |ENG       |2025-11-14 14:21:07.692407|NULL    |9006906596de1226a3d193c6dfd268d3e32372d08c4543bc19d5a455988aa1ac|
|615     |Albany           |New York       |United States |North America|Northeast      |US    |NY        |2025-11-14 14:21:07.692407|NULL    |c9b6664096efc1b8b

In [12]:
# # Step 1: Define schema with comments
# schema = sdt.StructType([
#     sdt.StructField("city_staging_key", sdt.IntegerType(), True, {"comment": "Row ID within the staging table"}),
#     sdt.StructField("city", sdt.StringType(), True, {"comment": "Formal name of the city"}),
#     sdt.StructField("state_province", sdt.StringType(), True, {"comment": "State or province for this city"}),
#     sdt.StructField("country", sdt.StringType(), True, {"comment": "Country name"}),
#     sdt.StructField("continent", sdt.StringType(), True, {"comment": "Continent that this city is on"}),
#     sdt.StructField("sales_territory", sdt.StringType(), True, {"comment": "Sales territory for this StateProvince"}),
#     sdt.StructField("region", sdt.StringType(), True, {"comment": "Name of the region"}),
#     sdt.StructField("subregion", sdt.StringType(), True, {"comment": "Name of the subregion"}),
#     sdt.StructField("valid_from", sdt.TimestampType(), True, {"comment": "Valid from this date and time"}),
#     sdt.StructField("valid_to", sdt.TimestampType(), True, {"comment": "Valid until this date and time"})
# ])

# # Step 2: Create empty table with schema
# empty_df = spark.createDataFrame([], schema)

# empty_df.writeTo("staging.Integration.City") \
#     .using("iceberg") \
#     .tableProperty("comment", "Staging table for city-level integration data") \
#     .partitionedBy("country", "state_province") \
#     .createOrReplace()

# # Step 5: Write to Iceberg table
# joined_output.writeTo("staging.Integration.City") \
#     .using("iceberg") \
#     .append()




In [13]:
spark.stop()